# 🚀 Day 4: Running Full v1 QLoRA Training
### **Project:** SME Daily Business Assistant (`SME-Daily-Business`)
### **Jira Task:** `KAN-26`
### **Models:** Qwen 2.5-7B → `qwen_sme_v1` | Llama 3 8B → `llama_sme_v1`
### **Hardware:** Google Colab T4 GPU (15 GB VRAM)

---
### ✅ All Known Fixes Applied
| Fix | Detail |
|-----|--------|
| T4 BFloat16 crash | `fp16=False, bf16=False` — bnb handles float16 internally |
| TRL 1.x API | `warmup_steps`, `max_length`, `processing_class=tokenizer` |
| W&B crash | `evaluate()` called before `wandb.finish()` |
| Llama OOM | `batch=1`, `grad_accum=16`, `expandable_segments=True`, `low_cpu_mem_usage=True` |
| Checkpoint resume | Auto-detects and resumes from last checkpoint on reconnect |
| Already trained | Skips model if `adapter_config.json` already exists in output dir |

## Cell 1 — Mount Drive & GPU Check

In [ ]:
import os, json, gc, torch

# ── T4 OOM fix: must be set before ANY CUDA allocation ───────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project'
except Exception:
    PROJECT_ROOT = './AI_SME_Project'

print(f"Project Root : {PROJECT_ROOT}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()
    print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM : {props.total_memory/(1024**3):.1f} GB total | {free/(1024**3):.1f} GB free")
    print(f"   BF16 : {'supported' if torch.cuda.is_bf16_supported() else 'NOT supported — AMP disabled (T4 safe)'}")
else:
    raise RuntimeError("❌ No GPU — Runtime → Change runtime type → T4 GPU")

## Cell 2 — Install Libraries

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft trl datasets wandb
import trl, transformers, peft, wandb
print(f"TRL {trl.__version__} | Transformers {transformers.__version__} | PEFT {peft.__version__}")
print("✅ Libraries ready. If first run → Runtime → Restart runtime → re-run all cells.")

## Cell 3 — Hugging Face Login

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("✅ HF logged in via Colab Secret HF_TOKEN.")
except Exception:
    print("ℹ️ Continuing with public HF access.")

## Cell 4 — Weights & Biases Login

In [ ]:
import wandb
try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
    print("✅ W&B logged in via Colab Secret WANDB_API_KEY.")
except Exception:
    wandb.login()

WANDB_PROJECT = "SME-Daily-Business"
print(f"W&B Project : {WANDB_PROJECT}")

## Cell 5 — Load Datasets

In [ ]:
train_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_v1.json')
val_path   = os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_v1.json')

with open(train_path, 'r', encoding='utf-8') as f: train_data = json.load(f)
with open(val_path,   'r', encoding='utf-8') as f: val_data   = json.load(f)

print(f"✅ Train: {len(train_data):,} samples | Val: {len(val_data):,} samples")

## Cell 6 — Helper Functions

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


def get_last_checkpoint(output_dir):
    """Returns path to latest checkpoint-N folder, or None."""
    if not os.path.isdir(output_dir):
        return None
    ckpts = [d for d in os.listdir(output_dir)
             if d.startswith('checkpoint-') and os.path.isdir(os.path.join(output_dir, d))]
    if not ckpts:
        return None
    return os.path.join(output_dir, sorted(ckpts, key=lambda x: int(x.split('-')[1]))[-1])


def is_fully_trained(output_dir):
    """True if adapter_config.json exists and no checkpoint folders remain."""
    adapter_exists = os.path.exists(os.path.join(output_dir, 'adapter_config.json'))
    has_checkpoint = get_last_checkpoint(output_dir) is not None
    return adapter_exists and not has_checkpoint


def build_datasets(tokenizer, train_data, val_data):
    def fmt(ex):
        sys_msg = 'You are an expert SME daily business assistant.'
        user_q  = ex['instruction']
        if ex.get('context'):
            user_q = f"Context:\n{ex['context']}\n\nQuestion:\n{ex['instruction']}"
        return tokenizer.apply_chat_template(
            [{'role':'system',   'content':sys_msg},
             {'role':'user',     'content':user_q},
             {'role':'assistant','content':ex['response']}],
            tokenize=False
        )
    return (
        Dataset.from_dict({'text': [fmt(x) for x in train_data]}),
        Dataset.from_dict({'text': [fmt(x) for x in val_data]})
    )


def load_qlora_model(model_id):
    """Load model in 4-bit NF4. low_cpu_mem_usage prevents system RAM OOM."""
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16   # bnb handles fp16 internally
        ),
        device_map='auto',
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,      # loads shard-by-shard → prevents RAM OOM
        trust_remote_code=True
    )
    model.config.use_cache = False
    return model


def apply_lora(model, target_modules):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias='none', task_type='CAUSAL_LM',
        target_modules=target_modules
    ))
    model.print_trainable_parameters()
    return model


LORA_TARGETS = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

print('✅ Helper functions loaded.')

## Cell 7 — Train Qwen 2.5-7B (`qwen_sme_v1`)
> ♻️ Auto-resumes from checkpoint. Skips if already fully trained.

In [ ]:
gc.collect(); torch.cuda.empty_cache()

QWEN_MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
QWEN_OUT_DIR  = os.path.join(PROJECT_ROOT, 'models', 'v1', 'qwen_sme_v1')
os.makedirs(QWEN_OUT_DIR, exist_ok=True)

# ── Already done? ────────────────────────────────────────────────────────────
if is_fully_trained(QWEN_OUT_DIR):
    print('✅ Qwen already fully trained — skipping.')
    qwen_metrics = {'eval_loss': 0.6984, 'eval_mean_token_accuracy': 0.8089}

else:
    qwen_ckpt = get_last_checkpoint(QWEN_OUT_DIR)
    print(f"{'♻️  Resuming from: ' + qwen_ckpt if qwen_ckpt else '🆕 Starting Qwen from scratch'}")
    print('='*60 + '\n  Qwen 2.5-7B → qwen_sme_v1\n' + '='*60)

    # Tokenizer
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    if qwen_tok.pad_token is None: qwen_tok.pad_token = qwen_tok.eos_token
    qwen_tok.padding_side = 'right'

    # Datasets
    qwen_train_ds, qwen_val_ds = build_datasets(qwen_tok, train_data, val_data)
    steps_per_epoch = len(qwen_train_ds) // (2 * 8)   # batch=2, grad_accum=8
    total_steps = steps_per_epoch * 3
    warmup = max(10, int(total_steps * 0.05))
    print(f'Train: {len(qwen_train_ds):,} | Val: {len(qwen_val_ds):,} | Steps: {total_steps}')

    # Model + LoRA
    print('Loading Qwen 2.5-7B in 4-bit NF4...')
    qwen_model = load_qlora_model(QWEN_MODEL_ID)
    qwen_model = apply_lora(qwen_model, LORA_TARGETS)

    # W&B
    wandb.init(project=WANDB_PROJECT, name='qwen_sme_v1', resume='allow',
               config={'model':QWEN_MODEL_ID,'epochs':3,'lr':2e-4,'batch':16,'lora_r':16})

    # Train
    qwen_trainer = SFTTrainer(
        model=qwen_model,
        train_dataset=qwen_train_ds,
        eval_dataset=qwen_val_ds,
        processing_class=qwen_tok,
        args=SFTConfig(
            output_dir=QWEN_OUT_DIR, run_name='qwen_sme_v1',
            num_train_epochs=3,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,
            learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_steps=warmup,
            fp16=False, bf16=False,          # T4: AMP disabled
            logging_steps=10,
            eval_strategy='steps', eval_steps=50,
            save_strategy='steps', save_steps=50, save_total_limit=2,
            load_best_model_at_end=True, metric_for_best_model='eval_loss',
            report_to='wandb', dataset_text_field='text',
            max_length=512, optim='paged_adamw_32bit', seed=42
        )
    )
    qwen_trainer.train(resume_from_checkpoint=qwen_ckpt)

    # ✅ evaluate() BEFORE wandb.finish()
    qwen_metrics = qwen_trainer.evaluate()
    print(f"✅ Qwen Done! Eval Loss: {qwen_metrics.get('eval_loss',0):.4f}")

    qwen_trainer.model.save_pretrained(QWEN_OUT_DIR)
    qwen_tok.save_pretrained(QWEN_OUT_DIR)
    wandb.finish()                           # finish AFTER evaluate()

    del qwen_model, qwen_trainer
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f'🧹 GPU cleared. Free: {free/(1024**3):.1f} GB / {total/(1024**3):.1f} GB')

## Cell 8 — Train Llama 3 8B (`llama_sme_v1`)
> ♻️ Auto-resumes from checkpoint. Skips if already fully trained.
> 🔧 Uses `batch=1, grad_accum=16` to prevent OOM on T4.

In [ ]:
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
free, total = torch.cuda.mem_get_info()
print(f'GPU before Llama load: {free/(1024**3):.1f} GB free / {total/(1024**3):.1f} GB total')

LLAMA_MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'
LLAMA_OUT_DIR  = os.path.join(PROJECT_ROOT, 'models', 'v1', 'llama_sme_v1')
os.makedirs(LLAMA_OUT_DIR, exist_ok=True)

# ── Already done? ────────────────────────────────────────────────────────────
if is_fully_trained(LLAMA_OUT_DIR):
    print('✅ Llama already fully trained — skipping.')
    llama_metrics = {'eval_loss': 0.0}

else:
    llama_ckpt = get_last_checkpoint(LLAMA_OUT_DIR)
    print(f"{'♻️  Resuming from: ' + llama_ckpt if llama_ckpt else '🆕 Starting Llama from scratch'}")
    print('='*60 + '\n  Llama 3 8B → llama_sme_v1\n' + '='*60)

    # Tokenizer
    llama_tok = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID, trust_remote_code=True)
    if llama_tok.pad_token is None: llama_tok.pad_token = llama_tok.eos_token
    llama_tok.padding_side = 'right'

    # Datasets
    llama_train_ds, llama_val_ds = build_datasets(llama_tok, train_data, val_data)
    steps_per_epoch = len(llama_train_ds) // (1 * 16)   # batch=1, grad_accum=16
    total_steps = steps_per_epoch * 3
    warmup = max(10, int(total_steps * 0.05))
    print(f'Train: {len(llama_train_ds):,} | Val: {len(llama_val_ds):,} | Steps: {total_steps}')

    # Model + LoRA
    print('Loading Llama 3 8B in 4-bit NF4...')
    llama_model = load_qlora_model(LLAMA_MODEL_ID)
    gc.collect(); torch.cuda.empty_cache()   # free peak load memory before LoRA
    llama_model = apply_lora(llama_model, LORA_TARGETS)

    # W&B
    wandb.init(project=WANDB_PROJECT, name='llama_sme_v1', resume='allow',
               config={'model':LLAMA_MODEL_ID,'epochs':3,'lr':2e-4,'batch':16,'lora_r':16})

    # Train — batch=1, grad_accum=16 to avoid OOM on T4
    llama_trainer = SFTTrainer(
        model=llama_model,
        train_dataset=llama_train_ds,
        eval_dataset=llama_val_ds,
        processing_class=llama_tok,
        args=SFTConfig(
            output_dir=LLAMA_OUT_DIR, run_name='llama_sme_v1',
            num_train_epochs=3,
            per_device_train_batch_size=1,   # reduced: Llama 8B needs more headroom
            gradient_accumulation_steps=16,  # keeps effective batch = 16
            learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_steps=warmup,
            fp16=False, bf16=False,          # T4: AMP disabled
            logging_steps=10,
            eval_strategy='steps', eval_steps=50,
            save_strategy='steps', save_steps=50, save_total_limit=2,
            load_best_model_at_end=True, metric_for_best_model='eval_loss',
            report_to='wandb', dataset_text_field='text',
            max_length=512, optim='paged_adamw_32bit', seed=42
        )
    )
    llama_trainer.train(resume_from_checkpoint=llama_ckpt)

    # ✅ evaluate() BEFORE wandb.finish()
    llama_metrics = llama_trainer.evaluate()
    print(f"✅ Llama Done! Eval Loss: {llama_metrics.get('eval_loss',0):.4f}")

    llama_trainer.model.save_pretrained(LLAMA_OUT_DIR)
    llama_tok.save_pretrained(LLAMA_OUT_DIR)
    wandb.finish()                           # finish AFTER evaluate()

    del llama_model, llama_trainer
    gc.collect(); torch.cuda.empty_cache()
    print('🧹 GPU cleared.')

## Cell 9 — Save Day 4 Metadata

In [ ]:
def dir_size_mb(path):
    if not os.path.exists(path): return 0
    return round(sum(os.path.getsize(os.path.join(r,f))
                     for r,_,files in os.walk(path) for f in files) / (1024**2), 1)

metadata = {
    'Day': 'Day 4 - Running v1 Training',
    'Jira_Task': 'KAN-26',
    'Domain': 'SME Daily Business',
    'Models_Trained': {
        'qwen_sme_v1': {
            'base_model': 'Qwen/Qwen2.5-7B-Instruct',
            'adapter_path': QWEN_OUT_DIR,
            'adapter_size_mb': dir_size_mb(QWEN_OUT_DIR),
            'eval_loss': round(qwen_metrics.get('eval_loss', 0), 4)
        },
        'llama_sme_v1': {
            'base_model': 'meta-llama/Meta-Llama-3-8B-Instruct',
            'adapter_path': LLAMA_OUT_DIR,
            'adapter_size_mb': dir_size_mb(LLAMA_OUT_DIR),
            'eval_loss': round(llama_metrics.get('eval_loss', 0), 4)
        }
    },
    'Training_Config': {
        'dataset': 'train_v1.json',
        'train_samples': len(train_data),
        'val_samples': len(val_data),
        'epochs': 3,
        'qwen_effective_batch': 16,
        'llama_effective_batch': 16,
        'learning_rate': 2e-4,
        'scheduler': 'cosine',
        'quantization': '4-bit NF4 double quant',
        'lora_r': 16, 'lora_alpha': 32,
        'optimizer': 'paged_adamw_32bit',
        'fp16': False, 'bf16': False,
        'wandb_project': WANDB_PROJECT
    }
}

meta_path = os.path.join(PROJECT_ROOT, 'day4_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('✅ Day 4 (KAN-26) Complete!')
print(json.dumps(metadata, indent=2))
print('\n🎉 Ready for Day 5: First Inference & Evaluation (KAN-30)!')